# Phase 5.2: EEGNet

The standard reference architecture for EEG deep learning. ~2K parameters, purpose-built for EEG signals using two compact-conv tricks: depthwise convolutions (each input filter learns its own spatial pattern, no mixing) and separable convolutions (depthwise temporal + pointwise channel mixing).

**Reference:**

Lawhern et al. (2018). "EEGNet: A Compact Convolutional Neural Network for EEG-based Brain-Computer Interfaces."
https://arxiv.org/abs/1611.08024

**Configuration here: EEGNet-8,2**

- F1 = 8           [number of temporal filters in block 1]
- D  = 2           [depth multiplier (each temporal filter -> D spatial)]
- F2 = F1 * D = 16 [number of separable filters in block 2]

**Settings learned from Phase 5.1b:**
- Class weights are essential
- Monitor val_acc, not val_loss (decouples when loss is class-weighted)


In [1]:

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

from phase5_utils import (
    load_phase2_data, make_subject_split, make_loaders,
    train_model, evaluate_model, count_parameters,
    PHASE5_DIR, SEED,
)

torch.manual_seed(SEED)
np.random.seed(SEED)

# EEGNet architecture

Compact CNN for EEG. Three blocks total, ~2K params for our (32, 640) input.
 
* Block 1: Temporal conv + Depthwise spatial conv
    - learns F1 temporal filters, then D spatial filters per temporal filter. So each frequency-pattern gets its own dedicated spatial patterns (rather than sharing them across frequencies).
 
* Block 2: Separable conv (depthwise temporal + pointwise 1x1 mix)
    - learns slower temporal abstractions, then mixes them across channels with a tiny 1x1 conv (a "learned channel reweighting").
 
* Out:     Flatten + Linear

In [2]:

"""
Input:  (B, 1, n_channels=32, n_samples=640)
Output: (B, n_classes)
"""

class EEGNet(nn.Module):
 
    def __init__(
        self,
        n_classes=2,
        n_channels=32,
        n_samples=640,
        F1=8,                # temporal filters
        D=2,                 # depth multiplier (spatial filters per temporal)
        F2=None,             # separable conv output filters (default F1*D)
        kernel_length=64,    # temporal kernel size (samples). 64@128Hz = 500ms.
        dropout=0.5,
    ):
        super().__init__()
        if F2 is None:
            F2 = F1 * D


        # BLOCK 1a: Temporal convolution
        # ============================================================
        # Standard Conv2d with kernel (1, 64). Acts like a learnable bank
        # of band-pass filters. Each of F1=8 output channels learns a
        # different temporal pattern, independently across all 32 EEG channels.
        # padding=(0, 32) keeps the time dimension roughly the same.

        self.conv_temporal = nn.Conv2d(
            in_channels=1,
            out_channels=F1,
            kernel_size=(1, kernel_length),
            padding=(0, kernel_length // 2),
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(F1)
 

        # BLOCK 1b: Depthwise spatial convolution  ← THE KEY TRICK
        # ============================================================
        # Normally a Conv2d(8, 16, kernel=(32,1)) would have 8*16*32 = 4096
        # weights, because each of 16 outputs is a weighted sum of ALL 8 inputs.
        #
        # With groups=F1=8, we split the 8 input filters into 8 separate groups
        # of 1. Each group becomes its own little Conv2d producing D=2 outputs.
        # So we have 8 *independent* mini-convs, each (1 → 2 outputs) with a
        # (32, 1) kernel. Total: 8 * 2 * 32 = 512 weights. 8x fewer parameters.
        #
        # Conceptually: each temporal filter (frequency-like pattern) gets its
        # OWN dedicated 2 spatial filters. The "alpha band" spatial pattern is
        # not forced to be a mixture of the "beta band" spatial pattern.

        self.depthwise_conv = nn.Conv2d(
            in_channels=F1,
            out_channels=F1 * D,
            kernel_size=(n_channels, 1),
            groups=F1,            # ← depthwise: F1 independent sub-convs
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))   # time downsample
        self.drop1 = nn.Dropout(dropout)


        # BLOCK 2: Separable convolution
        # ============================================================
        # A separable conv = (depthwise temporal) + (pointwise channel mix).
        # Equivalent expressiveness to a full Conv2d, fraction of the params.
 
        # Block 2a: depthwise temporal - each of the F1*D feature maps gets
        # its OWN temporal filter (no cross-channel mixing yet).
        # kernel (1, 16) at the post-pool time resolution.

        self.separable_depthwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F1 * D,
            kernel_size=(1, 16),
            padding=(0, 8),
            groups=F1 * D,       # depthwise (one filter per input channel)
            bias=False,
        )

        # Block 2b: pointwise 1x1 conv — mixes the F1*D feature maps into F2
        # output maps via a learned linear combination at each time step.
        # This is where cross-channel mixing finally happens (cheaply).

        self.separable_pointwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F2,
            kernel_size=(1, 1),
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))   # further time downsample
        self.drop2 = nn.Dropout(dropout)


        # Classifier
        # ============================================================
        # After all the pooling, time dim = floor((n_samples+1)/4) then
        # floor((...+1)/8). For n_samples=640 that's 20.
        # Feature vector size = F2 * 20.
        # We'll compute this dynamically via a dry forward pass below.

        self._n_features = self._compute_feature_size(n_channels, n_samples)
        self.classifier = nn.Linear(self._n_features, n_classes)
 
    def _compute_feature_size(self, n_channels, n_samples):
        """Dry-run a forward pass through everything but the classifier to
        figure out the flattened feature dim. Cleaner than hand-computing."""
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.conv_temporal(dummy)
            x = self.bn1(x)
            x = self.depthwise_conv(x)
            x = self.bn2(x)
            x = F.elu(x)
            x = self.pool1(x)
            x = self.separable_depthwise(x)
            x = self.separable_pointwise(x)
            x = self.bn3(x)
            x = F.elu(x)
            x = self.pool2(x)
            return int(x.flatten(1).shape[1])
 
    def forward(self, x):
        # x: (B, 1, 32, 640)
 
        # BLOCK 1 --------------------------------------------------
        x = self.conv_temporal(x)         # (B, 8, 32, 641)
        x = self.bn1(x)
        x = self.depthwise_conv(x)        # (B, 16, 1, 641)
        x = self.bn2(x)
        x = F.elu(x)
        x = self.pool1(x)                 # (B, 16, 1, 160)
        x = self.drop1(x)
 
        # BLOCK 2 --------------------------------------------------
        x = self.separable_depthwise(x)   # (B, 16, 1, 161)
        x = self.separable_pointwise(x)   # (B, 16, 1, 161)
        x = self.bn3(x)
        x = F.elu(x)
        x = self.pool2(x)                 # (B, 16, 1, 20)
        x = self.drop2(x)
 
        # CLASSIFIER --------------------------------------------------
        x = x.flatten(1)                  # (B, 320)
        x = self.classifier(x)            # (B, n_classes)
        return x

# Main Script

In [3]:

print("PHASE 5.2 — EEGNet-8,2 (binary classification)")
print("=" * 70)
 
 
# 1. Load data + split --------------------------------------------------

print("\n[1/5] Loading data and recreating Phase 5.0 split...")
X, y_binary, subjects = load_phase2_data(verbose=False)
train_idx, val_idx, test_idx, split_info = make_subject_split(subjects, n_train=32, n_val=4, n_test=4, seed=SEED)

train_loader, val_loader, test_loader = make_loaders(    X, y_binary, train_idx, val_idx, test_idx, batch_size=64)

print(f"  Train: {train_idx.sum()}  Val: {val_idx.sum()}  Test: {test_idx.sum()}")
 
 
# 2. Class weights (as in 5.1b) --------------------------------------------------

print("\n[2/5] Computing inverse-frequency class weights from TRAIN set...")
y_train = y_binary[train_idx]
class_counts = np.bincount(y_train, minlength=2)
n_total = class_counts.sum()
class_weights_np = n_total / (2 * class_counts)
class_weights = torch.FloatTensor(class_weights_np)
print(f"  Class weights: Relaxed={class_weights_np[0]:.3f}, Stress={class_weights_np[1]:.3f}")
 
criterion = nn.CrossEntropyLoss(weight=class_weights)
 
 
# 3. Build EEGNet --------------------------------------------------

print("\n[3/5] Building EEGNet-8,2...")
device = "cpu"
model = EEGNet(n_classes=2, n_channels=32, n_samples=640, F1=8, D=2, F2=16, kernel_length=64, dropout=0.5,).to(device)

n_params = count_parameters(model)
print(f"  Total trainable parameters: {n_params:,}")
print(f"  (Phase 5.1: 8,346  |  Phase 5.1b: 2,318  |  EEGNet: {n_params:,})")
print(f"  Flattened feature size: {model._n_features}")
 
with torch.no_grad():
    dummy = torch.zeros(2, 1, 32, 640)
    out = model(dummy)
    print(f"  Forward-pass sanity: input {tuple(dummy.shape)} → output {tuple(out.shape)}")
 
 
# 4. Train (with monitor='val_acc' since loss is class-weighted) --------------------------------------------------

print("\n[4/5] Training (class-weighted, monitor=val_acc)...")
start = time.time()
best_state, history = train_model(
    model, train_loader, val_loader,
    n_epochs=80, lr=1e-3, weight_decay=1e-4,
    patience=15,                 # more patience - smaller model, slower learning
    device=device, verbose=True,
    criterion=criterion,
    monitor="val_acc",           # save by best accuracy, not best loss
)
elapsed = time.time() - start
print(f"  Training time: {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
 
model.load_state_dict(best_state)
 
 
# 5. Evaluate --------------------------------------------------
print("\n[5/5] Evaluating on test set (4 unseen subjects)...")
results = evaluate_model(model, test_loader, device=device)
 
n0_test = int((y_binary[test_idx] == 0).sum())
n1_test = int((y_binary[test_idx] == 1).sum())
majority_baseline = max(n0_test, n1_test) / (n0_test + n1_test)
pred_counts = np.bincount(results["preds"], minlength=2)
pred_rate_stress = pred_counts[1] / pred_counts.sum()
 
print(f"  Test accuracy : {results['accuracy']:.4f}")
print(f"  Test F1 (pos) : {results['f1']:.4f}")
print(f"  Test F1 macro : {results['f1_macro']:.4f}")
print(f"  Confusion matrix:")
print(f"    {results['confusion_matrix']}")
print(f"  ---- Diagnostic checks ----")
print(f"  Model's prediction rate of 'Stress': {pred_rate_stress:.3f}  (true rate 0.396)")
print(f"  Best val_acc reached during training: {max(history['val_acc']):.4f}")
print(f"  ---- Reference points ----")
print(f"  Majority-class baseline on test  : {majority_baseline:.4f}")
print(f"  Phase 4B best (binary RF, LOSO)  : 0.5870")
print(f"  Phase 5.1  (no class weights)    : 0.4833")
print(f"  Phase 5.1b (smaller + weights)   : 0.5500")

PHASE 5.2 — EEGNet-8,2 (binary classification)

[1/5] Loading data and recreating Phase 5.0 split...
  Train: 1920  Val: 240  Test: 240

[2/5] Computing inverse-frequency class weights from TRAIN set...
  Class weights: Relaxed=1.136, Stress=0.893

[3/5] Building EEGNet-8,2...
  Total trainable parameters: 2,258
  (Phase 5.1: 8,346  |  Phase 5.1b: 2,318  |  EEGNet: 2,258)
  Flattened feature size: 320
  Forward-pass sanity: input (2, 1, 32, 640) → output (2, 2)

[4/5] Training (class-weighted, monitor=val_acc)...
  Epoch   1 | train_loss 0.7027 | val_loss 0.6903 | val_acc 0.5000 ←
  Epoch   2 | train_loss 0.6999 | val_loss 0.6878 | val_acc 0.5333 ←
  Epoch   3 | train_loss 0.6895 | val_loss 0.6884 | val_acc 0.5667 ←
  Epoch   4 | train_loss 0.6894 | val_loss 0.6888 | val_acc 0.5625 ←
  Epoch   5 | train_loss 0.6810 | val_loss 0.6952 | val_acc 0.5583 ←
  Epoch   6 | train_loss 0.6749 | val_loss 0.7066 | val_acc 0.5333 ←
  Epoch   7 | train_loss 0.6667 | val_loss 0.7040 | val_acc 0.5458 

# Save Plots and Results

In [4]:

out_dir = os.path.join(PHASE5_DIR, "phase5_2_eegnet")
os.makedirs(out_dir, exist_ok=True)
 
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="Train", color="#5DA5DA", marker="o", markersize=3)
axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#F15854", marker="o", markersize=3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Weighted CE loss")
axes[0].set_title("Loss curves (class-weighted)"); axes[0].legend(); axes[0].grid(alpha=0.3)
 
axes[1].plot(epochs, history["val_acc"], color="#60BD68", marker="o", markersize=3)
axes[1].axhline(majority_baseline, color="gray", linestyle="--", label=f"Majority baseline ({majority_baseline:.3f})")
axes[1].axhline(0.5870, color="purple", linestyle=":", label="Phase 4B RF (0.587)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Validation accuracy (monitored metric)")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1)
 
plt.suptitle(f"Phase 5.2 — EEGNet-8,2 ({n_params:,} params)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
plt.close()
 
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    results["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
    xticklabels=["Relaxed", "Stress"], yticklabels=["Relaxed", "Stress"], ax=ax,
)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Phase 5.2 (EEGNet) test confusion matrix\n"f"acc={results['accuracy']:.3f}, F1-macro={results['f1_macro']:.3f}")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
plt.close()
 
# JSON summary
results_summary = {
    "model": "EEGNet-8,2",
    "n_params": int(n_params),
    "training_time_sec": float(elapsed),
    "n_epochs_trained": len(history["train_loss"]),
    "test_accuracy": float(results["accuracy"]),
    "test_f1": float(results["f1"]),
    "test_f1_macro": float(results["f1_macro"]),
    "majority_baseline": float(majority_baseline),
    "predicted_stress_rate": float(pred_rate_stress),
    "best_val_acc": float(max(history["val_acc"])),
    "best_val_loss": float(min(history["val_loss"])),
    "confusion_matrix": results["confusion_matrix"].tolist(),
    "split_subjects": split_info,
    "hyperparameters": {
        "F1": 8, "D": 2, "F2": 16,
        "kernel_length": 64,
        "dropout": 0.5,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "max_epochs": 80,
        "patience": 15,
        "monitor": "val_acc",
        "class_weighted_loss": True,
    },
    "comparison": {
        "phase4b_rf_loso":    0.5870,
        "phase5_1_simple":    0.4833,
        "phase5_1b_simple_v2": 0.5500,
        "phase5_2_eegnet":    float(results["accuracy"]),
    },
}
with open(os.path.join(out_dir, "results.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
 
torch.save(best_state, os.path.join(out_dir, "eegnet_best.pt"))
 
print(f"\nSaved outputs to: {out_dir}")
print("=" * 70)
print("Phase 5.2 complete.")
print("=" * 70)


Saved outputs to: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\phase5_2_eegnet
Phase 5.2 complete.
